# 04 - Silver to Gold Load
## FoodLens Databricks Pipeline
### Purpose: Build Star Schema — Load Dimensions and Facts from Silver Delta Table

In [0]:
%run "./Util"

## Step 1: Imports and Read Silver Table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

silver = spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")
print(f"Silver rows loaded: {silver.count():,}")

Silver rows loaded: 1,179,288


## Step 2: Helper Functions
get_merge_stats — shows merge operation results
quarantine_fact_inspection — routes duplicate inspections to quarantine table

In [0]:
def get_merge_stats(DIM_TABLE):
    return spark.sql(f"DESCRIBE HISTORY {CATALOG}.{GOLD_SCHEMA}.{DIM_TABLE}") \
        .select("*") \
        .orderBy(F.col("version").desc()) \
        .limit(1) \
        .withColumn("Status",
            F.when(F.col("operationMetrics.numSourceRows") == 0, "No Data to Load")
            .when(F.col("operationMetrics.numTargetRowsInserted") != 0, "Data Inserted")
            .when(
                (F.col("operationMetrics.numSourceRows") != 0) &
                (F.col("operationMetrics.numTargetRowsInserted") == 0),
                "No Change in Data"
            ).otherwise("Failure")) \
        .selectExpr("timestamp", "job", "operation",
                    "operationMetrics.numOutputRows",
                    "operationMetrics.numSourceRows",
                    "operationMetrics.numTargetRowsInserted",
                    "Status")

def quarantine_fact_inspection(quarantine_df):
    quarantine_df.createOrReplaceTempView("staging_quarantine_inspection")
    spark.sql(f"""
        INSERT INTO {CATALOG}.{GOLD_SCHEMA}.{GOLD_QUARANTINE_INSPECTION}
        SELECT
            inspection_id, restaurant_id, restaurant_name, aka_name,
            CAST(inspection_date AS DATE),
            inspection_type, inspection_result, violation_score,
            violation_code, violation_points, pass_fail_flag,
            source_city, address, zip_code,
            quarantine_category, failed_column, source_system, pipeline_name,
            current_timestamp() AS _date_to_warehouse,
            current_timestamp() AS quarantine_timestamp
        FROM staging_quarantine_inspection
    """)

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


## Step 3: Load dim_date
Build date dimension from all distinct inspection dates in Silver.
MERGE ON date_key — insert only new dates.

In [0]:
dim_date = (
    silver
    .select("inspection_date")
    .distinct()
    .filter(F.col("inspection_date").isNotNull())
    .withColumn("date_key",           F.date_format("inspection_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("full_date",          F.col("inspection_date"))
    .withColumn("year",               F.year("inspection_date"))
    .withColumn("month_name",         F.date_format("inspection_date", "MMMM"))
    .withColumn("month_number",       F.month("inspection_date"))
    .withColumn("day",                F.date_format("inspection_date", "EEEE"))
    .withColumn("quarter",            F.concat(F.lit("Q"), F.quarter("inspection_date")))
    .withColumn("week_number",        F.weekofyear("inspection_date"))
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "date_key", "full_date", "year", "month_name",
        "month_number", "day", "quarter", "week_number",
        "_date_to_warehouse"
    )
)

dim_date.createOrReplaceTempView("staging_dim_date")

spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_DATE} AS target
    USING staging_dim_date AS source
    ON target.date_key = source.date_key
    WHEN NOT MATCHED THEN INSERT *
""")

print("✅ dim_date loaded!")
get_merge_stats(DIM_DATE).display()

✅ dim_date loaded!


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-21T20:26:05.000Z,null,MERGE,4518,4518,4518,Data Inserted


## Step 4: Load dim_violation_detail
Build violation dimension from all distinct violations in Silver.
MERGE ON violation_id — insert only new violations.

In [0]:
dim_violation = (
    silver
    .select("violation_id", "violation_code", "violation_description", "violation_detail")
    .distinct()
    .filter(
        F.col("violation_code").isNotNull() &
        (F.trim(F.col("violation_code")) != ""))
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "violation_id", "violation_code",
        "violation_description", "violation_detail", "_date_to_warehouse"
    )
)

dim_violation.createOrReplaceTempView("staging_dim_violation_detail")

spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL} AS target
    USING staging_dim_violation_detail AS source
    ON target.violation_id = source.violation_id
    WHEN NOT MATCHED THEN INSERT (
        violation_id, violation_code, violation_description,
        violation_detail, _date_to_warehouse
    )
    VALUES (
        source.violation_id, source.violation_code, source.violation_description,
        source.violation_detail, source._date_to_warehouse
    )
""")

print("✅ dim_violation_detail loaded!")
get_merge_stats(DIM_VIOLATION_DETAIL).display()

✅ dim_violation_detail loaded!


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-21T20:26:20.000Z,null,MERGE,911,911,911,Data Inserted


## Step 5: Load dim_restaurant (SCD Type 2)
Implements Slowly Changing Dimension Type 2 using UNION ALL NULL-key MERGE pattern.
- Expire old records when change_hash differs
- Insert new records from NULL merge_key branch

In [0]:
dim_restaurant = (
    silver
    .select("restaurant_id", "dba_name", "aka_name", "facility_type",
            "address", "source_city", "state", "zip_code", "change_hash")
    .distinct()
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .alias("s")
    .join(
        spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT}")
        .filter("is_current = true")
        .select("restaurant_id", "change_hash")
        .alias("t"),
        on=[
            F.col("s.restaurant_id") == F.col("t.restaurant_id"),
            F.col("s.change_hash") == F.col("t.change_hash")
        ],
        how="left_anti"
    )
)

dim_restaurant.createOrReplaceTempView("staging_dim_restaurant")
print(f"Restaurants to process: {dim_restaurant.count():,}")

Restaurants to process: 43,677


In [0]:
spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT} AS target
    USING (
        SELECT restaurant_id AS merge_key, * FROM staging_dim_restaurant
        UNION ALL
        SELECT NULL AS merge_key, * FROM staging_dim_restaurant
    ) AS source
    ON target.restaurant_id = source.merge_key
    AND target.is_current = true

    WHEN MATCHED AND target.change_hash <> source.change_hash THEN
        UPDATE SET
            target.is_current = false,
            target.effective_end_date = current_timestamp()

    WHEN NOT MATCHED AND source.merge_key IS NULL THEN
        INSERT (
            restaurant_id, dba_name, aka_name, facility_type,
            address, city, state, zip_code, change_hash,
            effective_start_date, effective_end_date,
            is_current, _date_to_warehouse
        )
        VALUES (
            source.restaurant_id, source.dba_name, source.aka_name,
            source.facility_type, source.address, source.source_city,
            source.state, source.zip_code, source.change_hash,
            current_timestamp(), TIMESTAMP '9999-12-31',
            true, source._date_to_warehouse
        )
""")

print("✅ dim_restaurant (SCD2) loaded!")
get_merge_stats(DIM_RESTAURANT).display()

✅ dim_restaurant (SCD2) loaded!


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-21T20:26:31.000Z,null,MERGE,43677,87354,43677,Data Inserted


## Step 6: Load fact_inspections
One row per inspection event.
Joins to dim_date and dim_restaurant to resolve surrogate keys.
MERGE ON inspection_id — safe append only.

In [0]:
dim_date_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_DATE}")
    .select("date_key", "full_date")
)

dim_rest_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT}")
    .select("restaurant_key", "restaurant_id", "dba_name", "aka_name", "facility_type", "change_hash")
    .distinct()
)

silver_df     = silver.alias("s")
dim_rest_lkp  = dim_rest_lkp.alias("r")
dim_date_lkp  = dim_date_lkp.alias("dt")

silver_df_agg = (
    silver
    .groupBy("inspection_id", "inspection_date", "restaurant_id")
    .agg(
        F.countDistinct("violation_id").alias("total_violation"),
        F.sum(F.coalesce(F.col("violation_points").cast(IntegerType()), F.lit(0))).alias("total_violation_points")
    )
).alias("agg")

fact_inspections = (
    silver_df_agg
    .join(silver.alias("s"), on=["inspection_id", "inspection_date", "restaurant_id"], how="left")
    .select(
        "s.inspection_id", "s.restaurant_id", "dba_name", "aka_name", "source_city",
        "s.inspection_date", "inspection_type", "risk_category", "facility_type",
        "address", "license_num", "zip_code", "violation_score",
        "inspection_result", "pass_fail_flag",
        "agg.total_violation", "agg.total_violation_points"
    )
    .distinct()
    .join(dim_date_lkp, F.col("s.inspection_date") == F.col("dt.full_date"), "left")
    .join(dim_rest_lkp,
          ((F.col("s.restaurant_id") == F.col("r.restaurant_id")) &
           (F.col("s.dba_name")      == F.col("r.dba_name"))      &
           (F.col("s.aka_name")      == F.col("r.aka_name"))),
          "left")
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "inspection_id", "restaurant_key",
        F.col("license_num").cast(IntegerType()),
        "risk_category",
        F.col("date_key").alias("inspection_date_key"),
        F.col("violation_score").alias("inspection_score"),
        "inspection_result",
        F.col("total_violation").cast(IntegerType()),
        F.col("total_violation_points").cast(IntegerType()),
        "inspection_type", "source_city", "pass_fail_flag",
        "_date_to_warehouse"
    )
)

fact_inspections.createOrReplaceTempView("staging_fact_inspections")
print(f"Fact inspections to load: {fact_inspections.count():,}")

Fact inspections to load: 279,471


In [0]:
spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS} target
    USING staging_fact_inspections source
    ON target.inspection_id = source.inspection_id
    WHEN NOT MATCHED THEN INSERT (
        inspection_id, restaurant_key, license_num, risk_category,
        inspection_date_key, inspection_score, inspection_result,
        total_violation, total_violation_points,
        inspection_type, source_city, pass_fail_flag, _date_to_warehouse
    )
    VALUES (
        source.inspection_id, source.restaurant_key, source.license_num,
        source.risk_category, source.inspection_date_key, source.inspection_score,
        source.inspection_result, source.total_violation, source.total_violation_points,
        source.inspection_type, source.source_city, source.pass_fail_flag,
        source._date_to_warehouse
    )
""")

print("✅ fact_inspections loaded!")
get_merge_stats(FACT_INSPECTIONS).display()

✅ fact_inspections loaded!


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-21T20:26:44.000Z,null,MERGE,279471,279471,279471,Data Inserted


## Step 7: Load fact_violations
One row per violation per inspection.
Joins to dim_violation_detail and fact_inspections to resolve surrogate keys.
MERGE ON (inspection_key, violation_code_key) — safe append only.

In [0]:
dim_viol_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL}")
    .select("violation_code_key", "violation_id")
)

w = Window.partitionBy("inspection_id").orderBy(F.col("inspection_score").desc())

fact_insp_lkp = (
    spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS}")
    .select("inspection_key", "inspection_id", "inspection_score")
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select("inspection_key", "inspection_id")
)

fact_violations = (
    silver
    .filter(
        F.col("violation_code").isNotNull() &
        (F.trim(F.col("violation_code")) != ""))
    .join(dim_viol_lkp, on=["violation_id"], how="left")
    .join(fact_insp_lkp, on="inspection_id", how="left")
    .withColumn("_date_to_warehouse", F.current_timestamp())
    .select(
        "inspection_key",
        "violation_code_key",
        F.col("violation_points"),
        F.col("is_critical_flag").alias("is_critical"),
        F.col("inspector_comments").alias("violation_comment"),
        "_date_to_warehouse"
    )
)

fact_violations.createOrReplaceTempView("staging_fact_violations")
print(f"Fact violations to load: {fact_violations.count():,}")

Fact violations to load: 1,179,288


In [0]:
spark.sql(f"""
    MERGE INTO {CATALOG}.{GOLD_SCHEMA}.{FACT_VIOLATIONS} target
    USING staging_fact_violations source
    ON target.inspection_key = source.inspection_key
    AND target.violation_code_key = source.violation_code_key
    WHEN NOT MATCHED THEN INSERT (
        inspection_key, violation_code_key, violation_points,
        is_critical, violation_comment, _date_to_warehouse
    )
    VALUES (
        source.inspection_key, source.violation_code_key, source.violation_points,
        source.is_critical, source.violation_comment, source._date_to_warehouse
    )
""")

print("✅ fact_violations loaded!")
get_merge_stats(FACT_VIOLATIONS).display()

✅ fact_violations loaded!


timestamp,job,operation,numOutputRows,numSourceRows,numTargetRowsInserted,Status
2026-04-21T20:26:54.000Z,null,MERGE,1179288,1179288,1179288,Data Inserted


In [0]:
## Step 8: Gold Layer Summary

In [0]:
print("=" * 55)
print("GOLD LAYER SUMMARY")
print("=" * 55)

tables = [DIM_DATE, DIM_RESTAURANT, DIM_VIOLATION_DETAIL, FACT_INSPECTIONS, FACT_VIOLATIONS]
for t in tables:
    count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{t}").count()
    print(f"{t:30} : {count:,} rows")

print("\n✅ Gold layer complete! Pipeline finished.")

GOLD LAYER SUMMARY
dim_date                       : 4,518 rows
dim_restaurant                 : 43,677 rows
dim_violation_detail           : 911 rows
fact_inspections               : 279,471 rows
fact_violations                : 1,179,288 rows

✅ Gold layer complete! Pipeline finished.
